# CGM & Wearables Report

Ambulatory Glucose Profile for the **launched** patient, sourced from the JupyterHealth
Exchange. This notebook is rendered by Voilà as the provider-facing app: the EHR launch
supplies the patient, the cells below compute and draw the report. Code input is hidden
in the rendered view (`strip_sources`), so only the report shows.

In [ ]:
# ======== LAYOUT TWEAKS (Voilà) ========
# Widen only the widget layer. Never restyle body/#rendered_cells/.jp-Notebook:
# Voilà's loader positions #rendered_cells with inline styles, and !important
# overrides there shift the whole page (learned the hard way).
from IPython.display import HTML, display
display(HTML("""<style>
.jupyter-widgets.widget-tab { width: 100% !important; max-width: 100% !important; }
.jp-OutputArea-output, .jp-RenderedHTMLCommon { max-width: 100% !important; }
.js-plotly-plot { width: 100% !important; }
</style>
<script>
/* Plotly widgets arrive late (after kernel execution) — watch for them and
   nudge a window resize so autosize re-measures the now-full-width container. */
const _nudge = () => window.dispatchEvent(new Event('resize'));
new MutationObserver((muts) => {
  for (const m of muts) for (const n of m.addedNodes) {
    if (n.nodeType === 1 && (n.classList?.contains('js-plotly-plot') || n.querySelector?.('.js-plotly-plot'))) {
      setTimeout(_nudge, 150); setTimeout(_nudge, 800);
    }
  }
}).observe(document.body, {childList: true, subtree: true});
window.__layout_nudge = true;
</script>"""))

# --- launch timing instrumentation: per-cell durations to container logs ---
import sys as _sys, time as _time
_ip = get_ipython()
_tm = {"t": None, "t0": _time.time()}
def _pre(info=None): _tm["t"] = _time.time()
def _post(result=None):
    if _tm["t"] is None: return
    try:
        head = (result.info.raw_cell or "").splitlines()[0][:70] if result is not None else ""
    except Exception:
        head = ""
    print(f"TIMING +{_time.time()-_tm['t0']:5.1f}s cell {_time.time()-_tm['t']:5.1f}s :: {head}",
          file=_sys.__stderr__, flush=True)
_ip.events.register("pre_run_cell", _pre)
_ip.events.register("post_run_cell", _post)


In [ ]:
# ============================================================================
# SCAFFOLD — launch context → JHE token → MRN → this patient's glucose frame.
# Usually untouched. Edit the ANALYTICS cells below, not this one.
# ============================================================================
import pandas as pd
from jupyterhealth_client import Code, tidy_observation
from provider_app import identity, launch_context, jhe_auth, patient_resolver

pd.options.mode.chained_assignment = None

import sys as _s2, time as _t2
def _mk(msg, t):
    print(f"TIMING-SCAFFOLD {msg} {_t2.time()-t:4.1f}s", file=_s2.__stderr__, flush=True)
_t = _t2.time(); ctx = launch_context.current(); _mk("EHR launch ctx + Patient fetch", _t)
_t = _t2.time(); client = jhe_auth.client_for_launch(ctx); _mk("JHE token exchange", _t)

# Resolve the launched patient in JHE and pull glucose. If the JHE account isn't
# authorized to see this patient (JupyterHealth RBAC), patient_resolver raises
# PatientNotInJHE — we catch it (and any other load error) so the dashboard shows a
# clean message and reveals NO patient info, instead of a traceback. None on success.
access_note = None
patient_id = None
df = pd.DataFrame()
raw_obs = []
try:
    _t = _t2.time(); patient_id = patient_resolver.resolve_patient(ctx.patient_mrn, client); _mk("resolve_patient scan", _t)
    # Same-person guard (family name + DOB vs the launched EHR patient): a wrong or
    # stale MRN mapping in JHE must fail CLOSED, not surface another patient's data.
    identity.assert_same_patient(ctx, client.get_patient(patient_id))
    # ONE fetch of the patient's observations; the showcase cells reuse raw_obs
    # instead of re-pulling, halving the JHE round-trips per launch.
    _t = _t2.time(); raw_obs = list(client.list_observations(patient_id=patient_id, limit=50_000)); _mk("observations fetch", _t)
    # Build the glucose frame directly: tidy_observation() calls pd.to_datetime per
    # scalar timestamp (~10k calls here, ~7s). One vectorized parse does the same job.
    _t = _t2.time()
    import base64 as _b64, json as _json
    _rows = []
    for _o in raw_obs:
        try:
            if _o["code"]["coding"][0]["code"] != "omh:blood-glucose:4.0":
                continue
            _b = _json.loads(_b64.b64decode(_o["valueAttachment"]["data"]))["body"]
            _rows.append((_b["blood_glucose"]["value"], _b["blood_glucose"].get("unit", "mg/dL"), _b["effective_time_frame"]["date_time"]))
        except (KeyError, IndexError, ValueError):
            continue
    df = pd.DataFrame(_rows, columns=["blood_glucose_value", "blood_glucose_unit", "_raw_dt"])
    if not df.empty:
        _ts = pd.to_datetime(df.pop("_raw_dt"), utc=True, format="ISO8601")
        df["effective_time_frame_date_time"] = _ts
        df["effective_time_frame_date_time_local"] = _ts.dt.tz_localize(None)
        df["code"] = "omh:blood-glucose:4.0"
    _mk("glucose frame (vectorized)", _t)
except patient_resolver.PatientNotInJHE:
    access_note = (
        "You don't have access to this patient's data in JupyterHealth. They may not be "
        "enrolled, or your JupyterHealth account isn't authorized to view them — access "
        "is governed by JupyterHealth, separately from your EHR permissions."
    )
except identity.IdentityError:
    access_note = (
        "Patient identity could not be verified, or does not match the launched "
        "patient — refusing to display data."
    )
except Exception as _e:
    access_note = f"Couldn't load JupyterHealth data — {type(_e).__name__}: {_e}"

# Tidy CGM frame, normalized to mg/dL (empty when there's no data / no access). JHE
# datasets label the unit variously ("mg/dL", "MGDL"); convert mmol/L (×18.0182).
if df.empty:
    cgm = df
else:
    cgm = df.loc[:, ["blood_glucose_value", "effective_time_frame_date_time_local"]].copy()
    _unit = df["blood_glucose_unit"].astype(str).str.upper().str.replace(r"[^A-Z]", "", regex=True)
    cgm.loc[_unit.values == "MMOLL", "blood_glucose_value"] *= 18.0182
    cgm["effective_time_frame_date_time_local"] = pd.to_datetime(
        cgm["effective_time_frame_date_time_local"]
    )
    cgm = cgm.sort_values("effective_time_frame_date_time_local")

In [ ]:
# ======== PATIENT HEADER ========
import requests as _rq
from IPython.display import Markdown, display


def md(text):
    display(Markdown(text))


try:
    me = client.get_user()
    practitioner = f"{me['firstName']} {me['lastName']} ({me['email']})"
except Exception:
    practitioner = "(unknown)"

if access_note:
    # JHE has NOT authorized this provider for this patient. Reveal nothing
    # patient-identifying on the JupyterHealth surface — only the access notice.
    md("# Patient Report")
    md(f"Signed in as **{practitioner}**")
    md(f"> 🔒 **{access_note}**")
else:
    # Authorized: demographics come from the EHR's FHIR Patient (system of record for
    # who the patient is). ctx carries the EHR token + FHIR base.
    try:
        _p = _rq.get(
            f"{ctx.fhir_base.rstrip('/')}/Patient/{ctx.fhir_patient_id}",
            headers={
                "Authorization": f"Bearer {ctx.access_token}",
                "Accept": "application/fhir+json",  # Epic serves XML by default
            },
            timeout=15,
        ).json()
        _nm = (_p.get("name") or [{}])[0]
        _name = (" ".join(_nm.get("given", [])) + " " + _nm.get("family", "")).strip() or "(unnamed)"
        _dob = _p.get("birthDate", "—")
        _sex = (_p.get("gender") or "—").title()
        _contact = next((t.get("value") for t in _p.get("telecom", []) if t.get("value")), "")
    except Exception:
        _name, _dob, _sex, _contact = "(demographics unavailable)", "—", "—", ""

    md(f"## {_name}")
    _line = f"**DOB** {_dob}  ·  **Sex** {_sex}  ·  **MRN** `{ctx.patient_mrn}`  ·  **JHE patient** {patient_id}"
    if _contact:
        _line += f"  ·  {_contact}"
    md(_line)
    md(f"Reviewing practitioner: **{practitioner}**")
    if cgm.empty:
        _by = {}
        for _o in raw_obs:
            try:
                _c = _o["code"]["coding"][0]["code"]
            except (KeyError, IndexError):
                continue
            _by[_c] = _by.get(_c, 0) + 1
        _parts = [f"{_by[c]} {label}" for c, label in (
            ("ieee:sleep-episode:1.0", "sleep nights"), ("ieee:physical-activity:1.0", "activity records"),
            ("omh:heart-rate:2.0", "heart-rate samples"), ("omh:oxygen-saturation:2.0", "SpO₂ nights")) if _by.get(c)]
        if _parts:
            md("> **No CGM data.** Wearable data on file: " + ", ".join(_parts) + " (Oura ring via Open Wearables). Summary below.")
        else:
            md("> **No glucose or wearable data for this patient.** They appear enrolled, but no observations were found.")
    else:
        span = (
            f"{cgm.effective_time_frame_date_time_local.min():%Y-%m-%d} → "
            f"{cgm.effective_time_frame_date_time_local.max():%Y-%m-%d}"
        )
        md(f"{len(cgm):,} glucose readings · {span}")

In [ ]:
# ======== GLYCEMIC METRICS (GMI, CV, Time-in-Range, GRI) ========
if not cgm.empty:
    g = cgm.blood_glucose_value
    total = len(g)
    mean_glucose = g.mean()
    cv = g.std() / mean_glucose * 100
    gmi = 3.31 + 0.02392 * mean_glucose

    vlow = (g < 54).mean() * 100
    low = ((g >= 54) & (g < 70)).mean() * 100
    target = ((g >= 70) & (g <= 180)).mean() * 100
    high = ((g > 180) & (g <= 250)).mean() * 100
    vhigh = (g > 250).mean() * 100

    # Glycemia Risk Index (0 best → 100 worst), capped at 100.
    gri = min(3.0 * (vlow + 0.8 * low) + 1.6 * (vhigh + 0.5 * high), 100)

    md(
        "| Metric | Value |\n|---|---|\n"
        f"| Mean glucose | {mean_glucose:.0f} mg/dL |\n"
        f"| Glucose Management Indicator (GMI) | {gmi:.1f}% |\n"
        f"| Coefficient of variation (CV) | {cv:.0f}% |\n"
        f"| Time in range (70–180) | {target:.0f}% |\n"
        f"| Glycemia Risk Index (GRI) | {gri:.0f} |"
    )

In [ ]:
# ======== AMBULATORY GLUCOSE PROFILE (AGP) ========
# All readings collapsed onto a single 24-hour day, shown as percentile bands.
if not cgm.empty:
    import matplotlib.pyplot as plt

    t = cgm.effective_time_frame_date_time_local
    cgm["minute_of_day"] = t.dt.hour * 60 + t.dt.minute
    cgm["time_bin"] = (cgm.minute_of_day // 5) * 5

    agp = (
        cgm.groupby("time_bin").blood_glucose_value
        .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
        .unstack()
    )
    agp.columns = ["p5", "p25", "p50", "p75", "p95"]
    agp = agp.reset_index()
    agp["hour"] = agp.time_bin / 60

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.axhspan(70, 180, color="#CDECCD", alpha=0.5, label="Target 70–180")
    ax.fill_between(agp.hour, agp.p5, agp.p95, color="#D8E3E7", alpha=0.6, label="5–95%")
    ax.fill_between(agp.hour, agp.p25, agp.p75, color="#B0C4DE", alpha=0.7, label="25–75%")
    ax.plot(agp.hour, agp.p50, color="green", lw=2, label="Median")
    ax.set_title("Ambulatory Glucose Profile (AGP)")
    ax.set_xlabel("Time of day")
    ax.set_ylabel("Glucose (mg/dL)")
    ax.set_xticks([0, 6, 12, 18, 24])
    ax.set_xticklabels(["12 AM", "6 AM", "12 PM", "6 PM", "12 AM"])
    ax.set_xlim(0, 24)
    ax.set_ylim(0, 300)
    ax.set_yticks([54, 70, 180, 250, 300])
    ax.legend(loc="upper right", ncol=2)
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# ======== TIME IN RANGE (stacked bar) ========
if not cgm.empty:
    bands = {
        "Very Low (<54)": (vlow, "#a00000"),
        "Low (54–69)": (low, "#f44336"),
        "Target (70–180)": (target, "#9ccc9c"),
        "High (181–250)": (high, "#fdbe85"),
        "Very High (>250)": (vhigh, "#fd8d3c"),
    }
    fig, ax = plt.subplots(figsize=(3, 6))
    bottom = 0
    for label, (pct, color) in bands.items():
        ax.bar(0, pct, bottom=bottom, color=color, width=0.6, edgecolor="white")
        if pct > 3:
            ax.text(0.45, bottom + pct / 2, f"{pct:.0f}%  {label}", va="center", fontsize=9)
        bottom += pct
    ax.set_ylim(0, 100)
    ax.set_xlim(-0.4, 2.2)
    ax.set_xticks([])
    ax.set_ylabel("% of readings")
    ax.set_title("Time in Range")
    plt.show()

<hr style="border:none;border-top:3px solid #d96a1f;margin:48px 0 8px">

# Wearable summary — sleep, overnight vitals, activity

*Consumer wearable (Oura ring) data delivered through the JupyterHealth Exchange. Reference ranges are general adult guidance for screening; the ring is not a diagnostic device, so confirm anything flagged clinically. Only signals the patient actually has are shown.*


In [ ]:
# === CGM SHOWCASE LIBRARY ===
# Inline, self-contained: parse OMH bodies -> aggregate -> Plotly figures + live fetch.
# Voila hides cell sources (strip_sources), so this defines the dashboard without showing code.
import base64
import json

import pandas as pd
import plotly.graph_objects as go

# OMH/IEEE codes verified live against JHE (patient 40006, study 30006).
_SIGNAL_CODES = {
    "sleep": ("ieee:sleep-episode:1.0", "omh:sleep-episode:1.1"),
    "activity": "ieee:physical-activity:1.0",
    "heart_rate": "omh:heart-rate:2.0",
    "respiratory_rate": "omh:respiratory-rate:2.0",
    "oxygen_saturation": "omh:oxygen-saturation:2.0",
}
_GLUCOSE_TIME = "effective_time_frame_date_time_local"
_GLUCOSE_VALUE = "blood_glucose_value"

_GLUCOSE = "#e8820c"
_SLEEP_FILL = "rgba(58,110,165,0.20)"
_ACT_LINE = "rgba(31,158,122,0.55)"
_ACT_FILL = "rgba(31,158,122,0.14)"
_BAND = "rgba(31,158,122,0.12)"


# ---- OMH body parsing ----
def _interval(body):
    return (body.get("effective_time_frame") or {}).get("time_interval") or {}


def _val(body, key):
    return (body.get(key) or {}).get("value")


def parse_sleep_episode(body):
    ti = _interval(body)
    start, end = ti.get("start_date_time"), ti.get("end_date_time")
    if not start or not end:
        return None
    start_ts = pd.Timestamp(start)
    # IEEE 1752 names the field sleep_efficiency_percentage; OMH 1.1 sleep_maintenance_efficiency_percentage.
    eff = _val(body, "sleep_efficiency_percentage")
    if eff is None:
        eff = _val(body, "sleep_maintenance_efficiency_percentage")
    return {"start": start_ts, "end": pd.Timestamp(end), "date": start_ts.strftime("%Y-%m-%d"),
            "tst_sec": _val(body, "total_sleep_time"), "eff": eff,
            "light_sec": _val(body, "light_sleep_duration"), "deep_sec": _val(body, "deep_sleep_duration"),
            "rem_sec": _val(body, "rem_sleep_duration"), "awake_sec": _val(body, "wake_after_sleep_onset")}


def parse_activity(body):
    ti = _interval(body)
    start, end = ti.get("start_date_time"), ti.get("end_date_time")
    if not start:
        return None
    start_ts = pd.Timestamp(start)
    # Daily summaries carry duration in minutes, workouts in seconds.
    dur = body.get("duration") or {}
    minutes = dur.get("value")
    if minutes is not None and dur.get("unit") == "sec":
        minutes = minutes / 60
    return {"date": start_ts.strftime("%Y-%m-%d"), "start": start_ts, "end": pd.Timestamp(end) if end else None,
            "name": body.get("activity_name") or "activity", "minutes": minutes,
            "steps": _val(body, "base_movement_quantity"),
            "kcal": _val(body, "kcal_burned"), "distance_m": _val(body, "distance"),
            "light_min": _val(body, "duration_light_activity"), "moderate_min": _val(body, "duration_moderate_activity"),
            "vigorous_min": _val(body, "duration_vigorous_activity")}


def parse_heart_rate_sample(body):
    t = (body.get("effective_time_frame") or {}).get("date_time")
    value = _val(body, "heart_rate")
    if not t or value is None:
        return None
    return {"time": pd.Timestamp(t), "value": value,
            "resting": body.get("temporal_relationship_to_sleep") == "during sleep"}


def parse_overnight_vital(body, value_key):
    # Oura vitals via Open Wearables carry a single date_time; older seeded data an overnight interval.
    start = _interval(body).get("start_date_time") or (body.get("effective_time_frame") or {}).get("date_time")
    value = _val(body, value_key)
    if not start or value is None:
        return None
    return {"date": pd.Timestamp(start).strftime("%Y-%m-%d"), "value": value,
            "resting": body.get("temporal_relationship_to_sleep") == "during sleep"}


# ---- Live fetch (single unfiltered call, bucket by code, decode base64 OMH body) ----
def _code_of(record):
    coding = (record.get("code") or {}).get("coding") or [{}]
    return coding[0].get("code")


def _decode_body(record):
    data = (record.get("valueAttachment") or {}).get("data")
    if not data:
        return None
    try:
        return json.loads(base64.b64decode(data)).get("body")
    except Exception:
        return None


def _observations_by_code(client, patient_id, raw=None):
    buckets = {}
    if raw is None:
        raw = client.list_observations(patient_id=patient_id, limit=50_000)
    for rec in raw:
        body = _decode_body(rec)
        if body is None:
            continue
        buckets.setdefault(_code_of(rec), []).append(body)
    return buckets


def fetch_signals(patient_id, client, raw_obs=None):
    try:
        by_code = _observations_by_code(client, patient_id, raw=raw_obs)
    except Exception:
        by_code = {}

    def bodies(name):
        codes = _SIGNAL_CODES[name]
        codes = codes if isinstance(codes, tuple) else (codes,)
        return [b for code in codes for b in by_code.get(code, [])]

    sleep = [r for b in bodies("sleep") if (r := parse_sleep_episode(b))]
    activity = [r for b in bodies("activity") if (r := parse_activity(b))]
    vitals = {}
    for short, name in (("hr", "heart_rate"), ("rr", "respiratory_rate"), ("spo2", "oxygen_saturation")):
        vitals[short] = [r for b in bodies(name) if (r := parse_overnight_vital(b, name))]
    hr_series = [r for b in bodies("heart_rate") if (r := parse_heart_rate_sample(b))]
    return {"sleep": sleep, "activity": activity, "vitals": vitals, "hr_series": hr_series}


# ---- Aggregation ----
def time_in_range(values, lo, hi):
    s = pd.Series(list(values), dtype="float64")
    if s.empty:
        return 0.0
    return float(((s >= lo) & (s <= hi)).mean() * 100)


def _glucose_frame(cgm):
    # The source column is wall-clock local time; we treat it as UTC for internal
    # consistency (the window boundaries below are built the same way). Renders match.
    if cgm is None or cgm.empty:
        return pd.DataFrame(columns=["time", "value"])
    return pd.DataFrame({
        "time": pd.to_datetime(cgm[_GLUCOSE_TIME], utc=True),
        "value": cgm[_GLUCOSE_VALUE].astype("float64"),
    }).sort_values("time").reset_index(drop=True)


def build_payload(cgm, sleep, activity, vitals, hr_series=None):
    glucose = _glucose_frame(cgm)
    # Parser dicts carry exactly these keys; columns= fixes order (empty-safe too).
    sleep_df = pd.DataFrame(sleep, columns=["date", "start", "end", "tst_sec", "eff",
                                            "light_sec", "deep_sec", "rem_sec", "awake_sec"])
    # Daily summaries and workouts share one code; the summary is the row with an activity_name of
    # "daily activity summary" (steps total), everything else is a workout.
    _act_cols = ["date", "start", "end", "name", "minutes", "steps", "kcal", "distance_m",
                 "light_min", "moderate_min", "vigorous_min"]
    all_act = pd.DataFrame(activity, columns=_act_cols)
    _is_summary = all_act["name"].eq("daily activity summary") if not all_act.empty else pd.Series(dtype="bool")
    act_df = all_act[_is_summary].reset_index(drop=True) if not all_act.empty else all_act
    workouts_df = all_act[~_is_summary].reset_index(drop=True) if not all_act.empty else all_act
    hr_df = pd.DataFrame(hr_series or [], columns=["time", "value", "resting"])
    if not hr_df.empty:
        hr_df = hr_df.sort_values("time").reset_index(drop=True)

    vit = {}
    for short, rows in (("hr", vitals.get("hr", [])), ("rr", vitals.get("rr", [])), ("spo2", vitals.get("spo2", []))):
        rows_df = pd.DataFrame(rows, columns=["date", "value", "resting"])
        # Heart rate arrives as a 5-minute sleep series plus one flagged resting value per night:
        # keep the resting rows when present, else the nightly minimum stands in for resting HR.
        if short == "hr" and not rows_df.empty and rows_df["resting"].any():
            rows_df = rows_df[rows_df["resting"]]
        if rows_df.empty:
            vit[short] = pd.Series(dtype="float64")
        else:
            vit[short] = rows_df.groupby("date")["value"].agg("min" if short == "hr" else "mean").astype("float64")
    vitals_df = pd.DataFrame(vit)
    if not vitals_df.empty:
        vitals_df.index.name = "date"

    if not act_df.empty:
        days = pd.to_datetime(act_df["date"], utc=True)
        start = days.min().normalize()
        end = days.max().normalize() + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)
    elif not glucose.empty:
        start, end = glucose["time"].min(), glucose["time"].max()
    elif not sleep_df.empty:
        start, end = sleep_df["start"].min(), sleep_df["end"].max()
    else:
        start = end = pd.Timestamp("1970-01-01T00:00:00Z")

    if not glucose.empty:
        glucose = glucose[(glucose["time"] >= start) & (glucose["time"] <= end)].reset_index(drop=True)

    return {"glucose": glucose, "sleep": sleep_df, "activity": act_df, "workouts": workouts_df,
            "vitals": vitals_df, "hr_series": hr_df,
            "window": {"start": start, "end": end},
            "has": {"glucose": not glucose.empty, "sleep": not sleep_df.empty,
                    "activity": not act_df.empty, "vitals": not vitals_df.empty,
                    "hr_series": not hr_df.empty, "workouts": not workouts_df.empty}}


def summary_stats(payload):
    g = payload["glucose"]["value"] if not payload["glucose"].empty else pd.Series(dtype="float64")
    act, sleep = payload["activity"], payload["sleep"]
    mean_glucose = float(g.mean()) if not g.empty else None
    cv = float(g.std() / g.mean() * 100) if (not g.empty and g.mean() != 0 and not pd.isna(g.std())) else None
    vitals = payload["vitals"]
    return {
        "mean_glucose": round(mean_glucose) if mean_glucose is not None else None,
        "tir": round(time_in_range(g, 70, 180)) if not g.empty else None,
        "cv": round(cv) if cv is not None else None,
        "avg_steps": round(float(act["steps"].mean())) if not act.empty and act["steps"].notna().any() else None,
        "avg_kcal": round(float(act["kcal"].mean())) if not act.empty and act["kcal"].notna().any() else None,
        "avg_sleep_eff": round(float(sleep["eff"].mean()), 1) if not sleep.empty and sleep["eff"].notna().any() else None,
        "avg_tst_min": round(float(sleep["tst_sec"].mean()) / 60) if not sleep.empty and sleep["tst_sec"].notna().any() else None,
        "avg_resting_hr": round(float(vitals["hr"].mean())) if "hr" in vitals and vitals["hr"].notna().any() else None,
        "avg_spo2": round(float(vitals["spo2"].mean()), 1) if "spo2" in vitals and vitals["spo2"].notna().any() else None,
        "avg_rr": round(float(vitals["rr"].mean()), 1) if "rr" in vitals and vitals["rr"].notna().any() else None,
        "workouts": int(len(payload["workouts"])) if "workouts" in payload else 0,
    }


# ---- Figures ----
def available_days(payload):
    g = payload["glucose"]
    if g.empty:
        return []
    return sorted(g["time"].dt.strftime("%Y-%m-%d").unique().tolist())


def overlay_figure(payload):
    g = payload["glucose"]
    fig = go.Figure()
    if not g.empty:
        # ISO strings, vectorized: Plotly passes strings straight through, avoiding
        # per-element Python datetime conversion (seconds-scale on thousands of points).
        _x = g["time"].dt.tz_localize(None).dt.strftime("%Y-%m-%dT%H:%M:%S")
        # lists, not arrays: FigureWidget serializes numpy ~100x slower than plain lists
        fig.add_trace(go.Scatter(x=_x.tolist(), y=g["value"].tolist(), name="Glucose",
                                 mode="lines", line=dict(color=_GLUCOSE, width=1.6)))
    if payload["has"]["activity"]:
        xs, ys = [], []
        for _, r in payload["activity"].dropna(subset=["kcal"]).iterrows():
            d0 = pd.Timestamp(r["date"])
            xs += [d0.isoformat(), (d0 + pd.Timedelta(hours=24)).isoformat()]
            ys += [r["kcal"], r["kcal"]]
        if xs:
            fig.add_trace(go.Scatter(x=xs, y=ys, name="Calories", yaxis="y2", mode="lines",
                                     line=dict(color=_ACT_LINE, width=1, shape="hv"),
                                     fill="tozeroy", fillcolor=_ACT_FILL))
    _shapes = []
    if payload["has"]["sleep"]:
        for _, s in payload["sleep"].iterrows():
            if pd.isna(s["start"]) or pd.isna(s["end"]):
                continue
            # tz-naive ISO strings: shape coords skip Plotly's tz handling entirely.
            # One shapes= assignment below instead of add_vrect per night — add_vrect
            # re-validates the whole shape list each call (quadratic).
            x0 = (s["start"].tz_localize(None) if s["start"].tzinfo else s["start"]).isoformat()
            x1 = (s["end"].tz_localize(None) if s["end"].tzinfo else s["end"]).isoformat()
            _shapes.append(dict(type="rect", xref="x", yref="paper", x0=x0, x1=x1, y0=0, y1=1,
                                fillcolor=_SLEEP_FILL, line_width=0, layer="below"))
    if payload["has"]["sleep"]:
        # vrect shapes don't show in the legend; add a labeled proxy so the bands read as sleep.
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers", name="Sleep window",
            marker=dict(size=11, color="rgba(58,110,165,0.55)", symbol="square"),
            hoverinfo="skip"))
    fig.update_layout(
        margin=dict(l=50, r=50, t=30, b=40), height=460, hovermode="x unified",
        xaxis=dict(type="date"),
        yaxis=dict(title="glucose mg/dL", range=[50, 300], visible=not g.empty),
        yaxis2=dict(title="kcal/day", overlaying="y", side="right", range=[0, 1800], showgrid=False),
        legend=dict(orientation="h"), template="plotly_white", shapes=_shapes)
    return fig


def day_figure(payload, day, lo, hi):
    g = payload["glucose"]
    d = g[g["time"].dt.strftime("%Y-%m-%d") == day] if not g.empty else g
    fig = go.Figure()
    tir = time_in_range(d["value"], lo, hi) if not d.empty else 0.0
    if not d.empty:
        fig.add_trace(go.Scatter(x=d["time"], y=d["value"], name="Glucose", mode="lines",
                                 line=dict(color=_GLUCOSE, width=2), fill="tozeroy",
                                 fillcolor="rgba(232,130,12,0.10)"))
        fig.add_hrect(y0=lo, y1=hi, fillcolor=_BAND, line_width=0, layer="below")
        if len(d) >= 2:  # low/peak only meaningful with >=2 readings
            lo_i, hi_i = d["value"].idxmin(), d["value"].idxmax()
            fig.add_trace(go.Scatter(
                x=[d.loc[lo_i, "time"], d.loc[hi_i, "time"]],
                y=[d.loc[lo_i, "value"], d.loc[hi_i, "value"]],
                mode="markers+text", text=["low", "peak"], textposition="bottom center",
                marker=dict(size=8, color=["#d64545", "#c77d1a"]), showlegend=False))
    fig.update_layout(
        title=f"{day} — {tir:.0f}% in range ({lo}–{hi})",
        margin=dict(l=50, r=20, t=40, b=40), height=420, hovermode="x unified",
        yaxis=dict(title="mg/dL", range=[50, max(190, (d['value'].max() + 15) if not d.empty else 190)]),
        template="plotly_white")
    return fig


def night_figures(payload):
    g = payload["glucose"]
    if g.empty:
        return []
    t = g["time"]
    hour = t.dt.hour + t.dt.minute / 60.0
    anchor = t.dt.strftime("%Y-%m-%d")
    morning = hour < 12
    anchor = anchor.where(~morning, (t - pd.Timedelta(days=1)).dt.strftime("%Y-%m-%d"))
    rel = hour.where(hour >= 18, hour + 24) - 18  # 18:00->0, 00:00->6, 06:00->12, noon->18
    work = pd.DataFrame({"anchor": anchor, "rel": rel, "value": g["value"], "hour": hour})
    work = work[(work["hour"] >= 18) | (work["hour"] < 12)]
    out = []
    for key in sorted(work["anchor"].unique()):
        seg = work[work["anchor"] == key].sort_values("rel")
        if len(seg) < 24:  # need ~2h+ of overnight coverage to be worth plotting
            continue
        fig = go.Figure(go.Scatter(x=seg["rel"], y=seg["value"], mode="lines",
                                   line=dict(color=_GLUCOSE, width=1.8), fill="tozeroy",
                                   fillcolor="rgba(232,130,12,0.10)"))
        fig.update_layout(height=160, margin=dict(l=30, r=8, t=8, b=20), showlegend=False,
                          template="plotly_white",
                          xaxis=dict(tickvals=[0, 6, 12, 18], ticktext=["18:00", "00:00", "06:00", "12:00"]),
                          yaxis=dict(range=[50, max(190, seg["value"].max() + 12)]))
        out.append((key, fig))
    return out


def sleep_figure(payload):
    """Stacked sleep stages per night (minutes) with time in bed as a line."""
    s = payload["sleep"].sort_values("start")
    fig = go.Figure()
    if s.empty:
        return fig
    x = s["date"].tolist()
    for col, name, color in (("deep_sec", "Deep", "#2f4b7c"), ("light_sec", "Light", "#6b8fc9"),
                             ("rem_sec", "REM", "#a3c4f3"), ("awake_sec", "Awake", "#e0a96d")):
        if s[col].notna().any():
            fig.add_trace(go.Bar(x=x, y=(s[col].fillna(0) / 60).round().tolist(), name=name, marker_color=color))
    in_bed = ((s["end"] - s["start"]).dt.total_seconds() / 60).round()
    fig.add_trace(go.Scatter(x=x, y=in_bed.tolist(), name="Time in bed", mode="lines+markers",
                             line=dict(color="#333", width=1.5, dash="dot")))
    fig.update_layout(barmode="stack", height=380, margin=dict(l=50, r=20, t=30, b=40),
                      yaxis=dict(title="minutes"), legend=dict(orientation="h"),
                      template="plotly_white", hovermode="x unified")
    return fig


def vitals_figure(payload):
    """Nightly resting heart rate, SpO2 and respiratory rate as small stacked panels."""
    from plotly.subplots import make_subplots
    v = payload["vitals"].sort_index()
    panels = [(k, t, c) for k, t, c in (("hr", "Resting HR (bpm)", "#c0392b"), ("spo2", "SpO₂ (%)", "#2980b9"),
                                        ("rr", "Respiratory rate (br/min)", "#27ae60"))
              if k in v.columns and v[k].notna().any()]
    fig = make_subplots(rows=max(len(panels), 1), cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=[t for _, t, _ in panels])
    for i, (k, t, c) in enumerate(panels, start=1):
        fig.add_trace(go.Scatter(x=v.index.tolist(), y=v[k].round(1).tolist(), name=t, mode="lines+markers",
                                 line=dict(color=c, width=2)), row=i, col=1)
    fig.update_layout(height=140 * max(len(panels), 1) + 60, margin=dict(l=50, r=20, t=40, b=40),
                      showlegend=False, template="plotly_white", hovermode="x unified")
    return fig




# ---- Clinical framing: reference ranges, per-night table, flags ----
import os as _os

# Wall-clock zone for bedtime / wake columns. Oura times arrive in UTC; the ring itself has no
# patient time zone in the exported record, so the deployment sets DISPLAY_TZ.
_DISPLAY_TZ = _os.environ.get("DISPLAY_TZ", "America/Los_Angeles")

# General adult screening guidance (not diagnostic thresholds): total sleep 7-9 h, sleep efficiency
# >=85 %, deep 13-23 % and REM 20-25 % of sleep, WASO <=30 min, mean overnight SpO2 >=95 % (<94 % flagged),
# nocturnal respiratory rate 12-20 /min, >=7,000 steps/day. Resting HR is judged against the patient's own
# 7-night mean (+5 bpm flagged) rather than a population cut-off.
_REF = {
    "tst_min": (420, 540), "eff": (85, None), "deep_pct": (13, 23), "rem_pct": (20, 25), "waso_min": (None, 30),
    "spo2": (95, None), "rr": (12, 20), "steps": (7000, None),
}
# How far outside the band still reads as amber rather than red.
_MARGIN = {"tst_min": 45, "eff": 5, "deep_pct": 5, "rem_pct": 5, "waso_min": 20, "spo2": 1, "rr": 2, "steps": 2000}
_OK, _WARN, _ALERT = "#2e7d32", "#c77d1a", "#c62828"


def _status(value, ref, warn_margin=0.0):
    """'ok' inside the reference band, 'warn' just outside it, 'alert' well outside; None when no value."""
    if value is None or pd.isna(value):
        return None
    lo, hi = ref
    if lo is not None and value < lo:
        return "warn" if value >= lo - warn_margin else "alert"
    if hi is not None and value > hi:
        return "warn" if value <= hi + warn_margin else "alert"
    return "ok"


def _local(ts):
    if ts is None or pd.isna(ts):
        return None
    ts = pd.Timestamp(ts)
    return (ts.tz_convert(_DISPLAY_TZ) if ts.tzinfo else ts.tz_localize("UTC").tz_convert(_DISPLAY_TZ))


def _hm(minutes):
    if minutes is None or pd.isna(minutes):
        return "—"
    minutes = int(round(minutes))
    return f"{minutes // 60}h {minutes % 60:02d}m"


def nightly_table(payload):
    """One row per night joining sleep, overnight vitals and that day's steps (by wake date)."""
    s = payload["sleep"]
    if s.empty:
        return pd.DataFrame()
    rows = []
    for _, n in s.sort_values("start").iterrows():
        in_bed_min = (n["end"] - n["start"]).total_seconds() / 60
        tst_min = n["tst_sec"] / 60 if pd.notna(n["tst_sec"]) else None
        rows.append({
            "date": n["date"],
            "bedtime": _local(n["start"]).strftime("%a %H:%M"), "wake": _local(n["end"]).strftime("%H:%M"),
            "in_bed_min": in_bed_min, "tst_min": tst_min,
            "eff": n["eff"],
            "deep_pct": (n["deep_sec"] / n["tst_sec"] * 100) if pd.notna(n["deep_sec"]) and n["tst_sec"] else None,
            "rem_pct": (n["rem_sec"] / n["tst_sec"] * 100) if pd.notna(n["rem_sec"]) and n["tst_sec"] else None,
            "waso_min": n["awake_sec"] / 60 if pd.notna(n["awake_sec"]) else None,
        })
    t = pd.DataFrame(rows).set_index("date")
    v = payload["vitals"]
    for col in ("hr", "spo2", "rr"):
        t[col] = v[col].reindex(t.index) if col in v.columns else None
    a = payload["activity"]
    t["steps"] = a.set_index("date")["steps"].reindex(t.index) if not a.empty else None
    return t


def clinical_flags(payload):
    """Plain-language flags a clinician would want to see first, with the night/day they refer to."""
    t = nightly_table(payload)
    out = []
    if t.empty:
        return out
    avg_tst = t["tst_min"].mean()
    if pd.notna(avg_tst) and avg_tst < 420:
        out.append(("warn", f"Average sleep {_hm(avg_tst)} over {len(t)} nights, below the 7 h adult guidance; {int((t['tst_min'] < 420).sum())} of {len(t)} nights under 7 h"))
    short = t[t["tst_min"] < 360]
    if len(short):
        out.append(("alert", f"Short sleep (<6 h) on {len(short)} of {len(t)} nights: " + ", ".join(f"{d} ({_hm(m)})" for d, m in short["tst_min"].items())))
    poor = t[t["eff"] < 85]
    if len(poor):
        out.append(("warn", f"Sleep efficiency below 85 % on {len(poor)} nights: " + ", ".join(f"{d} ({e:.0f} %)" for d, e in poor["eff"].items())))
    if t["spo2"].notna().any():
        low = t[t["spo2"] < 94]
        if len(low):
            out.append(("alert", f"Mean overnight SpO₂ below 94 % on {len(low)} nights: " + ", ".join(f"{d} ({v:.1f} %)" for d, v in low["spo2"].items()) + " — consider screening for sleep-disordered breathing"))
        elif (t["spo2"] < 95).any():
            out.append(("warn", "Mean overnight SpO₂ between 94 and 95 % on " + ", ".join(t.index[t["spo2"] < 95]) + " — borderline, watch for a trend"))
    if t["rr"].notna().any():
        odd = t[(t["rr"] < 12) | (t["rr"] > 20)]
        if len(odd):
            out.append(("warn", "Nocturnal respiratory rate outside 12–20 /min on " + ", ".join(f"{d} ({v:.1f})" for d, v in odd["rr"].items())))
    if t["hr"].notna().sum() >= 3:
        base = t["hr"].mean()
        high = t[t["hr"] > base + 5]
        if len(high):
            out.append(("warn", f"Resting heart rate ≥5 bpm above the 7-night mean ({base:.0f} bpm) on " + ", ".join(f"{d} ({v:.0f})" for d, v in high["hr"].items()) + " — often illness, alcohol or poor recovery"))
    bed = pd.Series([_local(ts) for ts in payload["sleep"]["start"]])
    if len(bed) >= 3:
        mins = pd.Series([(b.hour * 60 + b.minute + (1440 if b.hour < 12 else 0)) for b in bed])
        if mins.std() > 60:
            out.append(("warn", f"Irregular bedtime: night-to-night spread ±{mins.std():.0f} min (consistent timing supports sleep quality)"))
    if t["steps"].notna().any():
        low_days = t[t["steps"] < 5000]
        if len(low_days):
            out.append(("warn", f"Fewer than 5,000 steps on {len(low_days)} days: " + ", ".join(f"{d} ({int(v):,})" for d, v in low_days["steps"].items())))
    if not out:
        out.append(("ok", "No flags: sleep duration, efficiency, overnight SpO₂, breathing rate, resting heart rate and activity were within the reference bands on every night recorded."))
    return out


def nightly_table_html(payload):
    t = nightly_table(payload)
    if t.empty:
        return ""
    cols = [("bedtime", "Bedtime", None, None), ("wake", "Wake", None, None), ("in_bed_min", "In bed", _hm, None),
            ("tst_min", "Sleep", _hm, "tst_min"), ("eff", "Efficiency", lambda v: f"{v:.0f} %", "eff"),
            ("deep_pct", "Deep", lambda v: f"{v:.0f} %", "deep_pct"), ("rem_pct", "REM", lambda v: f"{v:.0f} %", "rem_pct"),
            ("waso_min", "Awake", lambda v: f"{v:.0f} min", "waso_min"), ("hr", "Resting HR", lambda v: f"{v:.0f} bpm", None),
            ("spo2", "SpO₂", lambda v: f"{v:.1f} %", "spo2"), ("rr", "Resp. rate", lambda v: f"{v:.1f} /min", "rr"),
            ("steps", "Steps", lambda v: f"{int(v):,}", "steps")]
    hr_base = t["hr"].mean() if t["hr"].notna().sum() >= 3 else None
    head = "<tr><th style='text-align:left'>Night</th>" + "".join(f"<th>{lab}</th>" for _, lab, _, _ in cols) + "</tr>"
    body = ""
    for date, r in t.iterrows():
        cells = ""
        for key, _, fmt, ref in cols:
            v = r.get(key)
            txt = "—" if v is None or (not isinstance(v, str) and pd.isna(v)) else (fmt(v) if fmt else v)
            status = _status(v, _REF[ref], _MARGIN[ref]) if ref else None
            if key == "hr" and hr_base is not None and pd.notna(v):
                status = "warn" if v > hr_base + 5 else "ok"
            bg = {"warn": "#fff4e0", "alert": "#fde2e1"}.get(status, "")
            cells += f"<td style='text-align:center;padding:4px 8px;background:{bg}'>{txt}</td>"
        body += f"<tr><td style='padding:4px 8px;font-weight:600'>{date}</td>{cells}</tr>"
    legend = ("<div style='font-size:11px;color:#6b7a85;margin-top:4px'>Reference bands: sleep 7–9 h · efficiency ≥85 % · deep 13–23 % · REM 20–25 % · awake ≤30 min · "
              "SpO₂ ≥95 % · resp. rate 12–20 /min · steps ≥7,000 · resting HR vs own 7-night mean. Amber = just outside, red = well outside.</div>")
    return (f"<table style='border-collapse:collapse;font-size:13px;width:100%'><thead style='background:#f3f1ec'>{head}</thead>"
            f"<tbody>{body}</tbody></table>{legend}")


def flags_html(flags):
    icon = {"alert": "🔴", "warn": "🟠", "ok": "🟢"}
    return "<ul style='padding-left:18px;margin:6px 0'>" + "".join(
        f"<li style='margin:3px 0'>{icon[level]} {text}</li>" for level, text in flags) + "</ul>"


def activity_figure(payload):
    """Daily steps against the 7,000-step guidance, workouts as markers."""
    a = payload["activity"].sort_values("date")
    fig = go.Figure()
    if a.empty:
        return fig
    colors = [_OK if (s or 0) >= 7000 else (_WARN if (s or 0) >= 5000 else _ALERT) for s in a["steps"]]
    fig.add_trace(go.Bar(x=a["date"].tolist(), y=a["steps"].tolist(), name="Steps", marker_color=colors,
                         text=[f"{int(s):,}" if pd.notna(s) else "" for s in a["steps"]], textposition="outside"))
    fig.add_hline(y=7000, line=dict(color="#6b7a85", dash="dot", width=1), annotation_text="7,000 steps", annotation_position="top left")
    w = payload["workouts"]
    if not w.empty:
        w = w.sort_values("start")
        fig.add_trace(go.Scatter(x=w["date"].tolist(), y=[400] * len(w), mode="markers", name="Workout",
                                 marker=dict(symbol="star", size=13, color="#1f4e79"),
                                 text=[f"{n}, {m:.0f} min" for n, m in zip(w["name"], w["minutes"].fillna(0))],
                                 hovertemplate="%{text}<extra></extra>"))
    ymax = max(float(a["steps"].max()) * 1.18, 8000)
    fig.update_layout(height=360, margin=dict(l=50, r=20, t=30, b=40), yaxis=dict(title="steps / day", range=[0, ymax]),
                      legend=dict(orientation="h"), template="plotly_white", hovermode="x unified")
    return fig


def night_hr_figures(payload):
    """Overnight heart-rate curve per night from the 5-minute samples inside each sleep window."""
    hr = payload["hr_series"]
    s = payload["sleep"].sort_values("start")
    if hr.empty or s.empty:
        return []
    out = []
    for _, n in s.iterrows():
        seg = hr[(hr["time"] >= n["start"]) & (hr["time"] <= n["end"]) & ~hr["resting"]]
        if len(seg) < 12:
            continue
        rel = (seg["time"] - n["start"]).dt.total_seconds() / 3600
        fig = go.Figure(go.Scatter(x=rel.round(2).tolist(), y=seg["value"].tolist(), mode="lines",
                                   line=dict(color="#c0392b", width=1.6), fill="tozeroy", fillcolor="rgba(192,57,43,0.08)"))
        lowest = float(seg["value"].min())
        fig.add_hline(y=lowest, line=dict(color="#6b7a85", dash="dot", width=1), annotation_text=f"low {lowest:.0f}", annotation_position="bottom right")
        fig.update_layout(height=170, margin=dict(l=30, r=8, t=8, b=22), showlegend=False, template="plotly_white",
                          xaxis=dict(title="hours after bedtime", dtick=2), yaxis=dict(range=[max(30, lowest - 10), float(seg["value"].max()) + 8]))
        out.append((n["date"], fig))
    return out


In [ ]:
# ======== SHOWCASE — fetch multi-signal data & build payload ========
showcase_payload = None
showcase_note = None
if access_note:
    showcase_note = "Showcase hidden — no authorized JupyterHealth access for this patient."
else:
    try:
        _sig = fetch_signals(patient_id, client, raw_obs=raw_obs or None)
        showcase_payload = build_payload(cgm, _sig["sleep"], _sig["activity"], _sig["vitals"], hr_series=_sig.get("hr_series"))
        if not any(showcase_payload["has"].values()):
            showcase_payload = None
            showcase_note = "No glucose or wearable data found for this patient."
    except Exception as _e:
        showcase_note = f"Couldn't build the showcase — {type(_e).__name__}: {_e}"

if showcase_note:
    md(f"> {showcase_note}")


In [ ]:
# ======== SHOWCASE — summary stats strip ========
if showcase_payload is not None:
    from IPython.display import HTML
    _s = summary_stats(showcase_payload)
    _t = nightly_table(showcase_payload)
    _nights = len(_t)
    # (label, value, unit, status, reference text)
    _cells = [
        ("Mean glucose", _s["mean_glucose"], "mg/dL", None, ""),
        ("Time in range", _s["tir"], "% 70–180", None, ""),
        ("Glucose CV", _s["cv"], "%", None, ""),
        ("Avg sleep", _hm(_s["avg_tst_min"]) if _s["avg_tst_min"] else None, f"per night · {_nights} nights", _status(_s["avg_tst_min"], _REF["tst_min"], 30), "ref 7–9 h"),
        ("Sleep efficiency", _s["avg_sleep_eff"], "%", _status(_s["avg_sleep_eff"], _REF["eff"], 5), "ref ≥85 %"),
        ("Resting HR", _s["avg_resting_hr"], "bpm", "ok" if _s["avg_resting_hr"] else None,
         f"nightly {_t['hr'].min():.0f}–{_t['hr'].max():.0f}" if not _t.empty and _t["hr"].notna().any() else ""),
        ("Overnight SpO₂", _s["avg_spo2"], "%", _status(_s["avg_spo2"], _REF["spo2"], 1), "ref ≥95 %"),
        ("Resp. rate", _s["avg_rr"], "/min", _status(_s["avg_rr"], _REF["rr"], 2), "ref 12–20"),
        ("Avg steps", f"{_s['avg_steps']:,}" if _s["avg_steps"] else None, "per day", _status(_s["avg_steps"], _REF["steps"], 2000), "ref ≥7,000"),
        ("Workouts", _s["workouts"] or None, "logged", None, ""),
    ]
    _colour = {"ok": _OK, "warn": _WARN, "alert": _ALERT}
    _html = "".join(
        f"<div style='flex:1;min-width:130px;padding:10px 14px;border:1px solid #e2ddd3;border-left:5px solid {_colour.get(st, '#e2ddd3')};border-radius:8px'>"
        f"<div style='font-size:22px;font-weight:600'>{v} <small style='color:#6b7a85;font-size:12px'>{u}</small></div>"
        f"<div style='font-family:monospace;font-size:10px;letter-spacing:.1em;text-transform:uppercase;color:#6b7a85'>{lab}</div>"
        f"<div style='font-size:11px;color:#6b7a85'>{ref}</div></div>"
        for (lab, v, u, st, ref) in _cells if v is not None
    )
    display(HTML(f"<div style='display:flex;gap:8px;flex-wrap:wrap;margin:12px 0'>{_html}</div>"))
    if _nights:
        md("### What stands out")
        display(HTML(flags_html(clinical_flags(showcase_payload))))
        md("### Night by night")
        display(HTML(nightly_table_html(showcase_payload)))


In [ ]:
# ======== SHOWCASE — interactive views (Master Overlay / Anatomy of a Day / Overnight) ========
if showcase_payload is not None:
    import ipywidgets as W
    import plotly.graph_objects as go
    from IPython.display import display
    _p = showcase_payload
    import time as _t3, sys as _s3
    def _q(msg, t):
        print(f"TIMING-TABS {msg} {_t3.time()-t:4.1f}s", file=_s3.__stderr__, flush=True)

    # Plain Plotly figures in Output widgets: constructing the first FigureWidget
    # pays a ~12s one-time widget-machinery init in the kernel; the browser-side
    # mime renderer keeps hover/zoom without any of that.
    def _fw(_f):
        _f.update_layout(autosize=True, width=None)
        _o = W.Output()
        with _o:
            display(_f)
        return _o

    # Tab 1 — Master Overlay
    _t = _t3.time(); overlay_w = _fw(overlay_figure(_p)); _q("overlay figure", _t)

    # Tab 2 — Anatomy of a Day (day picker + live normal-range slider)
    _days = available_days(_p)
    if _days:
        _dd = W.Dropdown(options=_days, value=_days[0], description="Day")
        _rng = W.IntRangeSlider(value=[70, 180], min=40, max=300, step=5,
                                description="Range", continuous_update=False)
        _day_out = W.Output()

        def _render_day(*_):
            _day_out.clear_output(wait=True)
            with _day_out:
                display(day_figure(_p, _dd.value, *_rng.value).update_layout(autosize=True))

        _dd.observe(_render_day, "value")
        _rng.observe(_render_day, "value")
        _t = _t3.time(); _render_day(); _q("anatomy first render", _t)
        anatomy = W.VBox([W.HBox([_dd, _rng]), _day_out])
    else:
        anatomy = W.HTML("<em>No daily glucose to drill into.</em>")

    # Tab 3 — Overnight small multiples with overnight vitals
    _t = _t3.time(); _nights = night_figures(_p)[-8:]; _q("night_figures", _t); _t = _t3.time()
    if _nights:
        _v = _p["vitals"]
        _cards = []
        for _label, _fig in _nights:
            _cap = ""
            if not _v.empty and _label in _v.index:
                _row = _v.loc[_label]
                _bits = [f"{_row[k]:.0f} {u}" for k, u in (("hr", "bpm"), ("rr", "rr"), ("spo2", "%SpO₂"))
                         if k in _v.columns and pd.notna(_row.get(k))]
                _cap = "  ·  ".join(_bits)
            _cards.append(W.VBox([W.HTML(f"<b>{_label} night</b><br><small>{_cap}</small>"),
                                  _fw(_fig)]))
        overnight = W.GridBox(_cards, layout=W.Layout(
            grid_template_columns="repeat(auto-fill, minmax(260px, 1fr))", grid_gap="14px"))
    else:
        overnight = W.HTML("<em>No overnight glucose windows.</em>")

    _panels = []
    if _p["has"]["sleep"]:
        _panels.append(("Sleep stages", _fw(sleep_figure(_p))))
    _hr_nights = night_hr_figures(_p)[-8:]
    if _hr_nights:
        _cards = [W.VBox([W.HTML(f"<b>{_label} night</b>"), _fw(_fig)]) for _label, _fig in _hr_nights]
        _panels.append(("Overnight heart rate", W.GridBox(_cards, layout=W.Layout(
            grid_template_columns="repeat(auto-fill, minmax(260px, 1fr))", grid_gap="14px"))))
    if _p["has"]["vitals"]:
        _panels.append(("Overnight vitals", _fw(vitals_figure(_p))))
    if _p["has"]["activity"]:
        _panels.append(("Activity", _fw(activity_figure(_p))))
    # The glucose views (overlay, day, overnight) only make sense with glucose; wearable-only patients skip them.
    if _p["has"]["glucose"]:
        _panels += [("Master Overlay", overlay_w), ("Anatomy of a Day", anatomy), ("Overnight", overnight)]
    _tabs = W.Tab(children=[w for _, w in _panels], layout=W.Layout(width="100%"))
    _q("cards + FigureWidgets + Tab", _t)
    for _i, (_title, _) in enumerate(_panels):
        _tabs.set_title(_i, _title)
    display(_tabs)